# Python to C++ Porting with Frontier LLMs

A notebook for porting Python code to high-performance C++ using frontier open-source LLMs via Groq's completely free API.

## What This Notebook Does

This notebook helps you **convert Python code to C++** automatically using state-of-the-art AI models. The generated C++ code is optimized for maximum runtime performance, making it ideal for computationally intensive tasks.

## Why Use This?

- **Performance Gains**: C++ can be 10-100x faster than Python for numerical computations
- **No GPU Required**: Runs on Groq's cloud API - no local GPU needed
- **Completely Free**: Groq's free tier requires no credit card
- **LLM-Based Compilation**: Uses AI to determine optimal compilation flags for your system
- **Multi-Model Support**: Compare results across different frontier models
- **Works Anywhere**: Local machine, Colab, or any environment with Python

## How It Works

1. **System Detection**: Analyzes your hardware (CPU, SIMD support, etc.)
2. **LLM Compiler Config**: Uses AI to determine optimal C++ compilation flags
3. **Code Porting**: Converts Python code to C++ using frontier LLMs
4. **Compilation**: Compiles the C++ code with optimized flags
5. **Benchmarking**: Compares Python vs C++ performance

## Key Features

- **Groq Free Tier**: No credit card required, rate-limited but generous
- **Frontier Models**: gpt-oss-20b, Qwen2.5-Coder-32B, DeepSeek-R1-Distill-Qwen-32B
- **Interactive UI**: Gradio interface for easy code conversion
- **Performance Comparison**: Side-by-side Python vs C++ results
- **Universal Compatibility**: Works locally, in Colab, or any Python environment

## Environment Setup

Import required libraries and set up the environment.

In [ ]:
# Import standard libraries
import os
import io
import sys
import json
import subprocess
from dotenv import load_dotenv
import gradio as gr
from IPython.display import Markdown, display
from openai import OpenAI
from groq import Groq

# Add helpers directory to path
sys.path.insert(0, './helpers')

# Import helper modules
from helpers.system_info import retrieve_system_info
from helpers.styles import CSS

system_info = retrieve_system_info()

print("Environment setup complete")
print(json.dumps(system_info, indent=2))

In [ ]:
# Load Groq API key from .env file
load_dotenv()
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print("Groq API key loaded from .env file")
else:
    print("GROQ_API_KEY not found in .env file")
    print("Please create a .env file with GROQ_API_KEY=your_key_here")

## Groq Model Configuration

**PLACEHOLDER MODELS** - These are placeholder models to be replaced with models selected from leaderboards.

Current placeholder models (all available on Groq):
- **gpt-oss-20b**: openai/gpt-oss-20b (21B params, 3.6B active, OpenAI open-weight, excellent reasoning)
- **Qwen2.5-Coder-32B**: qwen/qwen2.5-coder-32b (32B params, code-specialized, 5.5T code tokens)
- **DeepSeek-R1-Distill-Qwen-32B**: deepseek-ai/DeepSeek-R1-Distill-Qwen-32b (32B params, excellent reasoning and coding)

In [ ]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp 
and then execute it in the simplest way possible. Please reply with whether 
I need to install a C++ compiler to do this. If so, please provide the 
simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something 
like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. 
Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}
"""

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
response = groq.chat.completions.create(model="openai/gpt-oss-20b", messages=[{"role": "user", "content": message}])


display(Markdown(response.choices[0].message.content))

## Groq Model Configuration

Current models (all available on Groq):
- **gpt-oss-20b**: openai/gpt-oss-20b (21B params, 3.6B active, OpenAI open-weight, excellent reasoning)
- **Qwen2.5-Coder-32B**: qwen/qwen2.5-coder-32b (32B params, code-specialized, 5.5T code tokens)
- **DeepSeek-R1-Distill-Qwen-32B**: deepseek-ai/DeepSeek-R1-Distill-Qwen-32b (32B params, excellent reasoning and coding)

## Compiler Commands

**IMPORTANT**: Update these commands based on your system's C++ compiler.

For typical systems, the commands would be:

In [ ]:
# UPDATE THESE based on LLM response from previous cell
compile_command = [
    "clang++",
    "main.cpp",
    "-o", "main",
    "-O3",
    "-march=native",
    "-flto",
    "-pipe",
    "-stdlib=libc++",
]

run_command = ["./main"]

print(f"Compile command: {' '.join(compile_command)}")
print(f"Run command: {' '.join(run_command)}")

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python_code):
    return f"""
Port this Python code to C++ with the fastest possible implementation that 
produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled 
and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python_code}
```
"""

def messages_for(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python_code)}
    ]

def write_output(cpp_code):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp_code)

## Code Porting Functions

In [ ]:
def port(model, python):
    response = groq.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    return reply

## Python Execution Functions

In [ ]:
def run_python(code):
    """Execute Python code and capture output"""
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

## C++ Compilation and Execution

In [ ]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [ ]:
models = ["openai/gpt-oss-20b", "llama-3.3-70b-versatile"]

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Port from Python to C++") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label="C++ (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button("Port to C++", elem_classes=["convert-btn"])
        cpp_run = gr.Button("Run C++", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label="C++ result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True, debug=True)